# Airflow Learning Lab

**Mode:** fast exploration

This notebook is a first practical exposure to Apache Airflow. The goal is to understand the workflow mental model, inspect one small DAG, and leave with a concrete project direction.

You can read this notebook straight through before running a full Airflow environment. The support files in `mini_project/` are here to make the ideas concrete without turning setup into the main event.

## What Airflow Is For

Airflow helps you define and orchestrate workflows made of multiple steps. It is useful when a process needs structure: task order, repeatability, logs, and a clear record of what happened.

## Why It Exists

A plain script can run one thing. Airflow becomes useful when you need to describe **a flow of things**: what runs first, what depends on what, and what should happen on each run. The main beginner win is that the workflow becomes visible instead of implicit.

## Mental Model

- A **DAG** is the workflow recipe.
- A **task** is one step inside the recipe.
- **Dependencies** say which steps must finish before others can start.
- A **run** is one execution of that recipe.

For a first pass, ignore production deployment and think in terms of: *can I read this workflow and follow the path from input to output?*

In [ ]:
workflow = {
    "load_orders": [],
    "summarize_orders": ["load_orders"],
    "write_summary": ["summarize_orders"],
}

for step, deps in workflow.items():
    upstream = ", ".join(deps) if deps else "nothing upstream"
    print(f"{step:18} <- {upstream}")

## Setup With `uv`

Keep setup short. The notebook is still useful if you only read it first.

```bash
uv venv
source .venv/bin/activate
# Use the current stable Airflow version and matching constraints file
uv pip install "apache-airflow==<airflow-version>" \
  --constraint "https://raw.githubusercontent.com/apache/airflow/constraints-<airflow-version>/constraints-<python-version>.txt"
```

The exact version and constraints file should come from the official Airflow install docs. For this lab, the important thing is understanding the pattern, not memorizing the install command.

## First Minimal Example

The smallest useful Airflow example is a DAG with a few TaskFlow tasks. `@dag` defines the workflow, `@task` wraps normal Python functions, and the function-call order shows the dependencies.

Instead of pasting a huge code block here, inspect the real support file that powers this lab. The next cell uses a small path helper so the support files resolve cleanly from either the repo root or `examples/airflow/`.

In [ ]:
from pathlib import Path

def resolve_support_file(relative_path: str) -> Path:
    relative = Path(relative_path)
    roots = [Path.cwd(), *Path.cwd().parents]
    candidates = []
    seen = set()

    for root in roots:
        for candidate in (
            (root / relative).resolve(),
            (root / "examples" / "airflow" / relative).resolve(),
        ):
            candidate_str = str(candidate)
            if candidate_str not in seen:
                seen.add(candidate_str)
                candidates.append(candidate)

    for candidate in candidates:
        if candidate.exists():
            return candidate

    checked = "\n".join(f"- {path}" for path in candidates)
    raise FileNotFoundError(
        f"Unable to find '{relative_path}'.\n"
        f"Checked these candidate paths:\n{checked}\n"
        "Run the notebook from the repo root or examples/airflow/, "
        "or verify that the mini_project/ files are present."
    )


In [ ]:
dag_file = resolve_support_file("mini_project/dags/csv_pipeline.py")
print(dag_file.read_text())

## Core Concepts

The first concepts to keep in view are:

1. **DAG**: the workflow definition.
2. **Task**: one unit of work.
3. **Dependency**: the order between tasks.
4. **Schedule**: Airflow is built for repeatable runs, even if this lab uses manual triggering.
5. **Observability**: task states and logs are part of the product, not an afterthought.

## What To Ignore For Now

- executors
- Kubernetes
- production deployment choices
- plugins and deep customization
- advanced scheduling edge cases

In [ ]:
import csv

data_file = resolve_support_file("mini_project/data/raw_orders.csv")

with data_file.open() as handle:
    rows = list(csv.DictReader(handle))

rows[:3]

## Annotated Mini Experiment

The data work in this lab is intentionally boring. That is a feature, not a bug. You want the workflow structure to stay visible.

The DAG reads order rows, ignores cancelled orders, computes a tiny summary, and writes one output file. Reproducing that logic in plain Python makes the task boundaries easier to see before Airflow adds scheduling and orchestration on top.

In [ ]:
from collections import Counter

valid_rows = [row for row in rows if row["status"] != "cancelled"]
customer_counts = Counter(row["customer"] for row in valid_rows)

summary = {
    "raw_rows": len(rows),
    "kept_rows": len(valid_rows),
    "total_revenue": round(sum(float(row["amount"]) for row in valid_rows), 2),
    "top_customer": customer_counts.most_common(1)[0][0] if customer_counts else None,
}

summary

## Common Patterns And Caveats

- Keep tasks small and readable.
- Keep the DAG file focused on workflow structure, not giant business logic blocks.
- Airflow coordinates work; it should not hide what the work actually is.
- For a first project, manual triggering is fine. Do not start with production scheduling questions.

## Main Project Idea

Build one small CSV processing workflow with three tasks:

1. load raw order data
2. compute a summary
3. write the summary to disk

Inspect first:

- `mini_project/dags/csv_pipeline.py`
- `mini_project/data/raw_orders.csv`
- `mini_project/README.md`

This is a good first project because the Airflow-specific part is obvious: one DAG, three tasks, one visible flow.

## Next Experiments

- Add a validation task before the summary step.
- Split one task into two smaller tasks and decide whether that made the workflow clearer.
- Change the summary output from JSON to CSV and inspect what stays the same in the DAG.

## Suggested Next Questions For The Agent

- When should I use Airflow instead of cron plus plain Python scripts?
- What makes a task boundary good or bad in an Airflow DAG?
- How would this example change if one step needed retry behavior?
- What should I learn next after DAGs, tasks, and dependencies?

## Sources And Next Reading

Main official sources used for this lab:

- Airflow docs home: <https://airflow.apache.org/docs/>
- Core concepts: <https://airflow.apache.org/docs/apache-airflow/stable/core-concepts/index.html>
- Airflow 101 tutorial: <https://airflow.apache.org/docs/apache-airflow/stable/tutorial/fundamentals.html>
- TaskFlow tutorial: <https://airflow.apache.org/docs/apache-airflow/stable/tutorial/taskflow.html>
- Apache Airflow repository: <https://github.com/apache/airflow>

Read the TaskFlow tutorial next if you want to connect this notebook's tiny DAG to the official beginner examples.